In [ ]:
import os
os.environ["MONAI_USE_CUPY"] = "0"

import subprocess
subprocess.run(["pip", "uninstall", "-y", "cupy-cuda11x", "cupy-cuda12x", "cupy"],
               capture_output=True)


In [ ]:
import os
import random
import numpy as np

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
!pip uninstall -y monai monai-weekly
!pip install -q --no-cache-dir "monai-weekly[nibabel, tqdm]" einops matplotlib

import monai
import matplotlib.pyplot as plt
import torch

print(f"✅ MONAI version: {monai.__version__}")
print(f"✅ GPU available: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU'}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json
import time
from functools import partial

from monai.losses import DiceCELoss
from monai.inferers import sliding_window_inference
from monai import transforms
from monai.transforms import (
    AsDiscrete,
    Activations,
    Compose,
    LoadImaged,
    NormalizeIntensityd,
    RandFlipd,
    CropForegroundd,
    RandScaleIntensityd,
    RandShiftIntensityd,
    RandCropByPosNegLabeld,
    EnsureTyped,
    EnsureChannelFirstd,
    ConvertToMultiChannelBasedOnBratsClassesd,
    ConcatItemsd,    # get_loader_fixed
    DeleteItemsd,    # get_loader_fixed
)
from monai.config import print_config
from monai.metrics import DiceMetric
from monai.utils import MetricReduction
from monai.networks.nets import SwinUNETR
from monai.data import DataLoader, Dataset, CacheDataset, decollate_batch

print_config()
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU'}")

In [ ]:
import os
from datetime import datetime

base_results_path = "/content/drive/MyDrive/BraTS_Final_Model"

def get_unique_path(path):
    if not os.path.exists(path):
        return path
    counter = 1
    base, ext = os.path.splitext(path)
    new_path = f"{base}_{counter}{ext}"
    while os.path.exists(new_path):
        counter += 1
        new_path = f"{base}_{counter}{ext}"
    return new_path

current_date = datetime.now().strftime("%Y-%m-%d_%H-%M")
initial_root_dir = os.path.join(base_results_path, current_date)
root_dir = get_unique_path(initial_root_dir)
os.makedirs(root_dir, exist_ok=True)

best_model_path = os.path.join(root_dir, "best_model.pth")
model_save_path = root_dir

json_list = os.path.join(root_dir, "brats21_folds.json")



In [ ]:
import urllib.request

json_url = "https://developer.download.nvidia.com/assets/Clara/monai/tutorials/brats21_folds.json"
urllib.request.urlretrieve(json_url, json_list)


In [ ]:
import os
import json
import shutil
from google.colab import userdata

cache_data_dir = '/content/brats_final'

cache_exists = False
if os.path.exists(cache_data_dir):
    patients_cache = [f for f in os.listdir(cache_data_dir) if f.startswith('BraTS2021')]
    if len(patients_cache) > 0:
        cache_exists = True

if not cache_exists:

    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

    os.makedirs('/root/.kaggle', exist_ok=True)
    with open('/root/.kaggle/kaggle.json', 'w') as f:
        json.dump({"username": os.environ['KAGGLE_USERNAME'], "key": os.environ['KAGGLE_KEY']}, f)
    !chmod 600 /root/.kaggle/kaggle.json

    !rm -rf /content/brats_download {cache_data_dir}
    os.makedirs('/content/brats_download', exist_ok=True)
    os.makedirs(cache_data_dir, exist_ok=True)

    print("🚀 Downloading dataset from Kaggle...")
    !kaggle datasets download -d dschettler8845/brats-2021-task1 -p /content/brats_download --unzip

    tar_path = '/content/brats_download/BraTS2021_Training_Data.tar'
    if os.path.exists(tar_path):
        print("📦 Extracting TAR file...")
        !tar -xf {tar_path} -C {cache_data_dir}
        os.remove(tar_path)
    else:
        print("❌ TAR file not found!")

    inner_folder = os.path.join(cache_data_dir, 'BraTS2021_Training_Data')
    if os.path.exists(inner_folder):
        print(" Organizing folders...")
        for item in os.listdir(inner_folder):
            shutil.move(os.path.join(inner_folder, item), os.path.join(cache_data_dir, item))
        os.rmdir(inner_folder)

    !rm -rf /content/brats_download

patients = sorted([f for f in os.listdir(cache_data_dir) if f.startswith('BraTS2021')])
if len(patients) > 0:
    sample_files = os.listdir(os.path.join(cache_data_dir, patients[0]))

def datafold_read(datalist, basedir, fold=0, key="training"):
    with open(datalist) as f:
        json_data = json.load(f)
    json_data = json_data[key]

    for d in json_data:
        for k in d:
            if isinstance(d[k], list): # Image (FLAIR, T1, T1ce, T2)
                updated_paths = []
                for iv in d[k]:
                    filename = os.path.basename(iv)
                    patient_id = "_".join(filename.split("_")[:2])
                    actual_path = os.path.join(basedir, patient_id, filename)
                    updated_paths.append(actual_path)
                d[k] = updated_paths

            elif isinstance(d[k], str) and len(d[k]) > 0:
                if k in ["label", "image"]:
                    filename = os.path.basename(d[k])
                    patient_id = "_".join(filename.split("_")[:2])
                    d[k] = os.path.join(basedir, patient_id, filename)
                else:
                    d[k] = os.path.join(basedir, d[k])

    tr, val = [], []
    for d in json_data:
        if "fold" in d and d["fold"] == fold:
            val.append(d)
        else:
            tr.append(d)
    return tr, val


train_files, val_files = datafold_read(
    datalist=json_list,
    basedir=cache_data_dir,
    fold=1
)



In [ ]:
import os


if len(train_files) > 0:
    sample = train_files[0]

    for img_path in sample["image"]:
        exists = "✅" if os.path.exists(img_path) else "❌"
        print(f"  {exists} {os.path.basename(img_path)} -> ( {img_path})")

    label_exists = "✅" if os.path.exists(sample["label"]) else "❌"
    print(f"  {label_exists} {os.path.basename(sample['label'])} -> (  {sample['label']})")
else:
    print("❌ EMPTY")

In [ ]:
import gc
import torch
from monai import data, transforms
from monai.data import CacheDataset

def get_loader_fixed(batch_size, train_files, val_files, roi):
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    def split_modalities(files):
        return [{
            "flair": d["image"][0],
            "t1ce":  d["image"][1],
            "t1":    d["image"][2],
            "t2":    d["image"][3],
            "label": d["label"],
        } for d in files]

    safe_train = split_modalities(train_files)
    safe_val   = split_modalities(val_files)

    mods     = ["flair", "t1ce", "t1", "t2"]
    all_keys = mods + ["label"]

    train_transform = transforms.Compose([
        transforms.LoadImaged(keys=all_keys),
        transforms.EnsureChannelFirstd(keys=all_keys),
        transforms.ConcatItemsd(keys=mods, name="image", dim=0),
        transforms.DeleteItemsd(keys=mods),
        transforms.ConvertToMultiChannelBasedOnBratsClassesd(keys="label"),
        transforms.CropForegroundd(keys=["image", "label"], source_key="image"),
        transforms.SpatialPadd(keys=["image", "label"], spatial_size=roi),
        transforms.NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
        transforms.RandCropByPosNegLabeld(
            keys=["image", "label"],
            label_key="label",
            spatial_size=roi,
            pos=1,
            neg=1,
            num_samples=4,
            allow_smaller=True,
        ),
        transforms.RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
        transforms.RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1),
        transforms.RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=2),
        transforms.RandScaleIntensityd(keys="image", factors=0.1, prob=1.0),
        transforms.RandShiftIntensityd(keys="image", offsets=0.1, prob=1.0),
        transforms.EnsureTyped(keys=["image", "label"], dtype=torch.float32),
    ])

    val_transform = transforms.Compose([
        transforms.LoadImaged(keys=all_keys),
        transforms.EnsureChannelFirstd(keys=all_keys),
        transforms.ConcatItemsd(keys=mods, name="image", dim=0),
        transforms.DeleteItemsd(keys=mods),
        transforms.ConvertToMultiChannelBasedOnBratsClassesd(keys="label"),
        transforms.CropForegroundd(keys=["image", "label"], source_key="image"),
        transforms.SpatialPadd(keys=["image", "label"], spatial_size=roi),
        transforms.NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
        transforms.EnsureTyped(keys=["image", "label"], dtype=torch.float32),
    ])

    num_workers_train = 4
    num_workers_val   = 2

     train_ds = CacheDataset(
        data=safe_train,
        transform=train_transform,
        cache_rate=0.5,
        num_workers=num_workers_train,
    )
    val_ds = CacheDataset(
        data=safe_val,
        transform=val_transform,
        cache_rate=1.0,
        num_workers=num_workers_val,
    )

    train_loader = data.DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers_train,
        pin_memory=True,
        prefetch_factor=2,
    )
    val_loader = data.DataLoader(
        val_ds,
        batch_size=1,
        shuffle=False,
        num_workers=num_workers_val,
        pin_memory=True,
        prefetch_factor=2,
    )

    return train_loader, val_loader

In [ ]:

import psutil

def check_mem_readiness(num_train, num_val, cache_rate_train, cache_rate_val):
    available_ram_gb = psutil.virtual_memory().available / 1e9

    estimated_cache_size = (num_train * cache_rate_train + num_val * cache_rate_val) * 0.8
    overhead_gb = 2.0
    total_estimated_usage = estimated_cache_size + overhead_gb

    print(f"--- Memory Check ---")
    print(f"Available RAM:       {available_ram_gb:.2f} GB")
    print(f"Train files cached:  {int(num_train * cache_rate_train)}/{num_train}")
    print(f"Val files cached:    {int(num_val * cache_rate_val)}/{num_val}")
    print(f"Estimated usage:     {total_estimated_usage:.2f} GB")
    print(f"--------------------")

    if total_estimated_usage > available_ram_gb * 0.9:
        print(f"🔴 WARNING: High risk of OOM! cache_rate.")
    else:
        print(f"🟢 Memory looks safe. You can proceed!")




check_mem_readiness(
    num_train=len(train_files),
    num_val=len(val_files),
    cache_rate_train=0.5,
    cache_rate_val=1.0,
)

In [ ]:

# CELL 1 — Imports
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
!pip install monai einops -q

In [ ]:
#CELL 2
class SineLayer(nn.Module):
    def __init__(self, in_features, out_features, is_first=False, omega_0=30.0):
        super().__init__()
        self.omega_0  = omega_0
        self.is_first = is_first
        self.linear   = nn.Linear(in_features, out_features)
        self._init_weights()

    def _init_weights(self):
        with torch.no_grad():
            if self.is_first:
                bound = 1.0 / self.linear.weight.shape[1]
            else:
                bound = (6.0 / self.linear.weight.shape[1]) ** 0.5 / self.omega_0
            self.linear.weight.data.uniform_(-bound, bound)
            self.linear.bias.data.uniform_(-bound, bound)

    def forward(self, x):
        return torch.sin(self.omega_0 * self.linear(x))

In [ ]:
# CELL 3 — TumourCellDensityEstimator
class TumourCellDensityEstimator(nn.Module):
    def __init__(self, in_channels=192, spatial_size=16, hidden_dim=256):
        super().__init__()
        self.S = spatial_size ** 3  # 4096
        in_dim = self.S * 2

        self.mlp = nn.Sequential(
            SineLayer(in_dim, hidden_dim, is_first=True,  omega_0=30.0),
            SineLayer(hidden_dim, hidden_dim, is_first=False, omega_0=1.0),
            nn.Linear(hidden_dim, self.S)
        )

    def _single_forward(self, x_single, t_pervoxel):
        S = x_single.shape[0]
        x_input = torch.cat([x_single, t_pervoxel], dim=0)  # (2S,)
        out = self.mlp(x_input)                               # (S,)
        return torch.sigmoid(out)

    def forward(self, feature_map, t=0.1):
        feature_map = feature_map.float()
        B, C, H, W, D = feature_map.shape
        S = H * W * D

        x = feature_map.reshape(B * C, S)

        x = (x - x.mean(dim=-1, keepdim=True)) / (x.std(dim=-1, keepdim=True) + 1e-6)

        t_pervoxel = torch.full(
            (S,), float(t),
            dtype=torch.float32,
            device=feature_map.device,
            requires_grad=True
        )

        u_flat = torch.vmap(
            lambda x_i: self._single_forward(x_i, t_pervoxel)
        )(x)  # (B*C, S)

        u_hat = u_flat.reshape(B, C, H, W, D)

        return u_hat, t_pervoxel

In [ ]:
# CELL 4 — Laplacian Kernel
def get_laplacian_kernel(device):
    K = torch.zeros(1, 1, 3, 3, 3, device=device)

     K[0, 0, 1, 1, 0] =  1
    K[0, 0, 1, 1, 2] =  1
    K[0, 0, 1, 0, 1] =  1
    K[0, 0, 1, 2, 1] =  1
    K[0, 0, 0, 1, 1] =  1
    K[0, 0, 2, 1, 1] =  1

    K[0, 0, 1, 1, 1] = -6

    return K

In [ ]:
# CELL 5
class PDELoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, u_hat, du_dt, d, rho):
        B, C, H, W, D = u_hat.shape

        K = get_laplacian_kernel(u_hat.device)


        du = d * u_hat  # (B, C, H, W, D)

         du_in = du.reshape(B * C, 1, H, W, D)
        div_d_grad_u = F.conv3d(du_in, K, padding=1).reshape(B, C, H, W, D)

        proliferation = rho * u_hat * (1 - u_hat)

        residual = du_dt - div_d_grad_u - proliferation
        return (residual ** 2).mean()

In [ ]:
# CELL 6 — Boundary Condition Loss
class BoundaryConditionLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, u_hat, d):
        # u_hat: (B, C, H, W, D)
        u = u_hat.float()
        B, C, H, W, D = u.shape

        ux_0 = (u[:, :, 1, :, :]  - u[:, :, 0, :, :])  ** 2
        ux_H = (u[:, :, -1, :, :] - u[:, :, -2, :, :]) ** 2
        uy_0 = (u[:, :, :, 1, :]  - u[:, :, :, 0, :])  ** 2
        uy_W = (u[:, :, :, -1, :] - u[:, :, :, -2, :]) ** 2
        uz_0 = (u[:, :, :, :, 1]  - u[:, :, :, :, 0])  ** 2
        uz_D = (u[:, :, :, :, -1] - u[:, :, :, :, -2]) ** 2

        bc_loss = (
            ux_0.mean() + ux_H.mean() +
            uy_0.mean() + uy_W.mean() +
            uz_0.mean() + uz_D.mean()
        )

         if isinstance(d, torch.Tensor):
            d_scalar = d.mean()
        else:
            d_scalar = float(d)

        return d_scalar * bc_loss

In [ ]:
# CELL 7
class BiophysicsRegulariser(nn.Module):
    def __init__(self, in_channels=192, spatial_size=16,
                 hidden_dim=256, lambda1=1e-4, lambda2=1e-4):
        super().__init__()
        self.lambda1 = lambda1
        self.lambda2 = lambda2
        self.d_min,   self.d_max   = 0.02, 1.5
        self.rho_min, self.rho_max = 0.002, 0.2

        self.estimator = TumourCellDensityEstimator(
            in_channels=in_channels,
            spatial_size=spatial_size,
            hidden_dim=hidden_dim,
        )
        self.pde_loss = PDELoss()
        self.bc_loss  = BoundaryConditionLoss()

    def _sample_params_per_voxel(self, shape, device, dtype):
        d   = torch.empty(shape, device=device, dtype=dtype).uniform_(self.d_min, self.d_max)
        rho = torch.empty(shape, device=device, dtype=dtype).uniform_(self.rho_min, self.rho_max)
        return d, rho

    def forward(self, feature_map, seg_loss, t=0.1):
        u_hat, t_pervoxel = self.estimator(feature_map, t=t)

        # du/dt per-voxel
        du_dt = torch.autograd.grad(
            outputs=u_hat.sum(),
            inputs=t_pervoxel,
            create_graph=True,
            retain_graph=True,
            allow_unused=True
        )[0]

        if du_dt is None:
            du_dt = torch.zeros_like(t_pervoxel)

        # reshape  (B, C, H, W, D)
        B, C, H, W, D = u_hat.shape
        du_dt = du_dt.reshape(1, 1, H, W, D).expand(B, C, H, W, D)

        d, rho = self._sample_params_per_voxel(u_hat.shape, feature_map.device, feature_map.dtype)

        lpde = self.lambda1 * self.pde_loss(u_hat, du_dt, d=d, rho=rho)
        lbc  = self.lambda2 * self.bc_loss(u_hat, d=d)
        total_loss = seg_loss + lpde + lbc
        info = {
            "loss_seg"  : seg_loss.item(),
            "loss_pde"  : lpde.item(),
            "loss_bc"   : lbc.item(),
            "loss_total": total_loss.item(),
        }
        return total_loss, info

In [ ]:
original_model_path = "/content/drive/MyDrive/BraTS_Final_Model/2026-05-24_16-17/best_model.pth"

In [ ]:
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from monai.metrics import HausdorffDistanceMetric

 device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

 roi           = (128, 128, 128)
max_epochs    = 30
val_every     = 5
sw_batch_size = 4

 if 'hook_handle' in locals() or 'hook_handle' in globals():
    try:
        hook_handle.remove()
     except Exception:
        pass

 try:
    model = SwinUNETR(
        img_size=roi,
        in_channels=4,
        out_channels=3,
        feature_size=48,
        drop_rate=0.0,
        attn_drop_rate=0.0,
        dropout_path_rate=0.0,
        use_checkpoint=True,
    ).to(device)
 except TypeError:
    model = SwinUNETR(
        in_channels=4,
        out_channels=3,
        feature_size=48,
        use_checkpoint=True,
    ).to(device)

 checkpoint = torch.load(original_model_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["state_dict"])
BASELINE_DICE = 0.9011
print(f"✅  {checkpoint['epoch']} (best_acc: {BASELINE_DICE:.4f})")

 for name, param in model.named_parameters():
    if any(x in name for x in ["patch_embed", "layers", "encoder1", "encoder2",
                               "encoder3", "encoder4", "encoder5", "encoder6",
                               "encoder7", "encoder8", "encoder9"]) and not "encoder10" in name:
        param.requires_grad = False
    else:
        param.requires_grad = True

frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"🔒  : {frozen:,} | 🔓   (Decoder + Encoder10): {trainable:,}")

 loss_function = DiceCELoss(
    to_onehot_y=False,
    sigmoid=True,
    squared_pred=True,
    smooth_nr=0,
    smooth_dr=1e-5,
)
print("✅ DiceCELoss  ready!")

dice_metric = DiceMetric(
    include_background=True,
    reduction=MetricReduction.MEAN_BATCH,
    get_not_nans=True,
)

hausdorff_metric = HausdorffDistanceMetric(
    include_background=True,
    percentile=95,
    reduction=MetricReduction.MEAN_BATCH,
    get_not_nans=True,
    directed=False,
)

post_sigmoid = Activations(sigmoid=True)
post_pred    = AsDiscrete(argmax=False, threshold=0.5)

 feature_map_holder = {}
def hook_fn(module, input, output):
    feature_map_holder["bottleneck"] = torch.as_tensor(output, device=output.device).float()

hook_handle = model.swinViT.layers2[0].register_forward_hook(hook_fn)

biophy_regulariser = BiophysicsRegulariser(
    in_channels=192,
    spatial_size=16,
    hidden_dim=256,
    lambda1=1e-4,
    lambda2=1e-4,
).to(device)

 optimizer = AdamW(
    [
        {"params": [p for p in model.parameters() if p.requires_grad], "lr": 1e-5},
        {"params": [p for p in biophy_regulariser.estimator.parameters()], "lr": 1e-4},
    ],
    weight_decay=1e-5
)

 scheduler = CosineAnnealingLR(optimizer, T_max=max_epochs, eta_min=1e-6)

 model_inferer = partial(
    sliding_window_inference,
    roi_size=roi,
    sw_batch_size=sw_batch_size,
    predictor=model,
    overlap=0.75,
)



In [ ]:
import os
import numpy as np
import torch
from monai.data import decollate_batch

class AverageMeter:
    def __init__(self):
        self.reset()
    def reset(self):
        self.val   = 0
        self.avg   = 0
        self.sum   = 0
        self.count = 0
    def update(self, val, n=1):
        self.val    = val
        self.sum   += val * n
        self.count += n
        self.avg    = np.where(self.count > 0, self.sum / self.count, 0)


def save_checkpoint(model, biophy_regulariser, epoch, best_acc, optimizer, scheduler,
                    filename="best_model_finetuned.pth"):
    checkpoint = {
        "epoch"             : epoch,
        "state_dict"        : model.state_dict(),
        "biophy_state_dict" : biophy_regulariser.state_dict(),
        "best_acc"          : best_acc,
        "optimizer"         : optimizer.state_dict(),
        "scheduler"         : scheduler.state_dict(),
    }
    save_path = os.path.join(root_dir, filename)
    torch.save(checkpoint, save_path)
    print(f"💾  keep in  {epoch}: {save_path}")


def train_epoch(model, biophy_regulariser, loader, optimizer, epoch,
                loss_func, device, max_epochs, scaler=None):
    model.train()
    biophy_regulariser.train()

    run_loss = AverageMeter()
    run_pde  = AverageMeter()
    run_bc   = AverageMeter()

    for idx, batch_data in enumerate(loader):
        data   = batch_data["image"].to(device)
        target = batch_data["label"].to(device)

        optimizer.zero_grad(set_to_none=True)

        logits   = model(data)
        seg_loss = loss_func(logits.float(), target.float())

        feature_map = feature_map_holder["bottleneck"].float()

        t_step = (epoch / max(max_epochs - 1, 1)) * 0.9 + 0.1

        total_loss, info = biophy_regulariser(feature_map, seg_loss, t=t_step)

        total_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad] +
            list(biophy_regulariser.parameters()),
            max_norm=1.0
        )

        optimizer.step()

        run_loss.update(info["loss_total"], n=1)
        run_pde.update(info["loss_pde"],    n=1)
        run_bc.update(info["loss_bc"],      n=1)

        if idx % 10 == 0:
            with torch.no_grad():
                u_pred, _ = biophy_regulariser.estimator(
                    feature_map_holder["bottleneck"].float().detach(), t=t_step)
                u_std = u_pred.std().item()
            print(f"Epoch {epoch}/{max_epochs-1} [{idx}/{len(loader)}] | "
                  f"Total: {run_loss.val:.4f} | "
                  f"PDE: {run_pde.val:.6f} | "
                  f"BC: {run_bc.val:.6f} | "
                  f"u_std: {u_std:.4f} | t={t_step:.3f}")

    return float(run_loss.avg)


def val_epoch(model, loader, acc_func, hd_func, model_inferer,
              post_sigmoid, post_pred, device):
    model.eval()
    run_acc = AverageMeter()
    run_hd  = AverageMeter()

    with torch.no_grad():
        for batch_data in loader:
            data   = batch_data["image"].to(device)
            target = batch_data["label"].to(device)

            logits = model_inferer(data)

            val_labels_list    = decollate_batch(target)
            val_outputs_list   = decollate_batch(logits)
            val_output_convert = [
                post_pred(post_sigmoid(val_pred_tensor))
                for val_pred_tensor in val_outputs_list
            ]

            acc_func.reset()
            acc_func(y_pred=val_output_convert, y=val_labels_list)
            acc, not_nans = acc_func.aggregate()
            run_acc.update(acc.cpu().numpy(), n=not_nans.cpu().numpy())

            hd_func.reset()
            hd_func(y_pred=val_output_convert, y=val_labels_list)
            hd, hd_not_nans = hd_func.aggregate()
            run_hd.update(hd.cpu().numpy(), n=hd_not_nans.cpu().numpy())

            del data, target, logits, val_labels_list, val_outputs_list, val_output_convert

    return run_acc.avg, run_hd.avg


def trainer(model, biophy_regulariser, train_loader, val_loader, optimizer, loss_func,
            acc_func, hd_func, scheduler, device, max_epochs, model_inferer,
            post_sigmoid, post_pred, val_every=5):

    val_acc_max       = 0.0
    patience          = 10
    epochs_no_improve = 0
    stop_training     = False

    loss_epochs  = []
    trains_epoch = []
    all_losses   = []
    dices_tc, dices_wt, dices_et, dices_avg = [], [], [], []
    hd_tc, hd_wt, hd_et = [], [], []

    for epoch in range(max_epochs):
        print("-" * 20)
        print(f"Epoch: {epoch}/{max_epochs-1} | No improve: {epochs_no_improve}/{patience}")

        train_loss = train_epoch(
            model, biophy_regulariser, train_loader, optimizer,
            epoch, loss_func, device, max_epochs, scaler=None
        )
        all_losses.append(train_loss)

        if (epoch + 1) % val_every == 0 or epoch == 0:
            loss_epochs.append(train_loss)
            trains_epoch.append(epoch)

            val_acc, val_hd = val_epoch(
                model, val_loader, acc_func, hd_func,
                model_inferer, post_sigmoid, post_pred, device
            )

            tc  = float(val_acc[0])
            wt  = float(val_acc[1])
            et  = float(val_acc[2])
            htc = float(val_hd[0])
            hwt = float(val_hd[1])
            het = float(val_hd[2])
            val_avg_acc = float(np.mean(val_acc))

            dices_tc.append(tc)
            dices_wt.append(wt)
            dices_et.append(et)
            dices_avg.append(val_avg_acc)
            hd_tc.append(htc)
            hd_wt.append(hwt)
            hd_et.append(het)

            print(f"📊 Dice — Avg={val_avg_acc:.4f} | TC={tc:.4f} | WT={wt:.4f} | ET={et:.4f}")
            print(f"📐 HD95 — TC={htc:.2f}mm | WT={hwt:.2f}mm | ET={het:.2f}mm")

             if epoch >= 10 and val_avg_acc < 0.883:
                 stop_training = True

            if val_avg_acc > val_acc_max:
                 val_acc_max = val_avg_acc
                save_checkpoint(model, biophy_regulariser, epoch,
                                val_acc_max, optimizer, scheduler)
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1

            if epochs_no_improve >= patience:
                print(f"🛑 Early Stopping   {epoch} epochs  !")
                stop_training = True

        scheduler.step()

        if stop_training:
            break

    print(f"\n🏆  Best Dice: {val_acc_max:.4f}")
    return (val_acc_max, dices_tc, dices_wt, dices_et, dices_avg,
            hd_tc, hd_wt, hd_et, loss_epochs, trains_epoch, all_losses)

print("✅ Trainer ready!")

In [ ]:
train_loader, val_loader = get_loader_fixed(
    batch_size=1,
    train_files=train_files,
    val_files=val_files,
    roi=roi
)

check_data = next(iter(train_loader))
img_shape   = list(check_data['image'].shape)
label_shape = list(check_data['label'].shape)

 print(f"Image shape (MRI): {img_shape}")
print(f"Label shape (Mask): {label_shape}")
print("------------------------------------------")

assert img_shape[1] == 4, f"❌  it's not 4 it's :{img_shape[1]}"
assert label_shape[1] == 3, f"❌ {label_shape[1]}"
assert img_shape[0] == label_shape[0], "❌ "

for dim_idx, (actual, expected) in enumerate(zip(img_shape[2:], list(roi))):
    assert actual <= expected, f"❌   {dim_idx} ROI is bigger than: {actual} > {expected}"



In [ ]:
model.eval()
biophy_regulariser.eval()

 batch = next(iter(train_loader))
data = batch["image"].to(device)

 with torch.no_grad():
    _ = model(data)

 fm = feature_map_holder["bottleneck"].to(dtype=torch.float32)

 with torch.enable_grad():
     fm_grad = fm.requires_grad_(False)

     u_hat, t_ten = biophy_regulariser.estimator(fm_grad, t=0.1)

     grad = torch.autograd.grad(
        outputs=u_hat,
        inputs=t_ten,
        grad_outputs=torch.ones_like(u_hat),
        create_graph=False
    )[0]

     mean_grad_value = grad.mean().item()
    u_std_value = u_hat.std().item()


    if mean_grad_value == 0.0:
        print("🔴 Worning")
    else:
        print("🟢 Perfect ")

In [ ]:
#  Fine-tuning
(
    val_acc_max,
    dices_tc, dices_wt, dices_et, dices_avg,
    hd_tc, hd_wt, hd_et,
    loss_epochs, trains_epoch, all_losses,
) = trainer(
    model              = model,
    biophy_regulariser = biophy_regulariser,
    train_loader       = train_loader,
    val_loader         = val_loader,
    optimizer          = optimizer,
    loss_func          = loss_function,
    acc_func           = dice_metric,
    hd_func            = hausdorff_metric,
    scheduler          = scheduler,
    device             = device,
    max_epochs         = max_epochs,
    model_inferer      = model_inferer,
    post_sigmoid       = post_sigmoid,
    post_pred          = post_pred,
    val_every          = val_every,
)

print(f"\n{'='*40}")
print(f"✅ Fine-tuning finished!")
print(f"🏆 Best Average Dice: {val_acc_max:.4f}")
print(f"   Best TC: {max(dices_tc):.4f}")
print(f"   Best WT: {max(dices_wt):.4f}")
print(f"   Best ET: {max(dices_et):.4f}")
print(f"   Best HD95 TC: {min(hd_tc):.2f}mm")
print(f"   Best HD95 WT: {min(hd_wt):.2f}mm")
print(f"   Best HD95 ET: {min(hd_et):.2f}mm")
print(f"{'='*40}")

I thought mabey this CELLS are better but IDK

In [ ]:
#CELL 1 — Imports
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
!pip install monai einops -q

In [ ]:
#CELL 2
class SineLayer(nn.Module):
    def __init__(self, in_features, out_features, is_first=False, omega_0=30.0):
        super().__init__()
        self.omega_0  = omega_0
        self.is_first = is_first
        self.linear   = nn.Linear(in_features, out_features)
        self._init_weights()

    def _init_weights(self):
        with torch.no_grad():
            if self.is_first:
                bound = 1.0 / self.linear.weight.shape[1]
            else:
                bound = (6.0 / self.linear.weight.shape[1]) ** 0.5 / self.omega_0
            self.linear.weight.data.uniform_(-bound, bound)
            self.linear.bias.data.uniform_(-bound, bound)

    def forward(self, x):
        return torch.sin(self.omega_0 * self.linear(x))

In [ ]:
#CELL 3 — TumourCellDensityEstimator
class TumourCellDensityEstimator(nn.Module):
    def __init__(self, in_channels=192, spatial_size=16, hidden_dim=256):
        super().__init__()
        self.S = spatial_size ** 3
        in_dim = self.S * 2

        self.mlp = nn.Sequential(
            SineLayer(in_dim, hidden_dim, is_first=True,  omega_0=30.0),
            SineLayer(hidden_dim, hidden_dim, is_first=False, omega_0=1.0),
            nn.Linear(hidden_dim, self.S)
        )

    def forward(self, feature_map, t=0.1):
        feature_map = feature_map.float()
        B, C, H, W, D = feature_map.shape
        S = H * W * D

        x = feature_map.reshape(B * C, S)

        x = (x - x.mean(dim=-1, keepdim=True)) / (x.std(dim=-1, keepdim=True) + 1e-6)

        t_tensor = torch.tensor(
            float(t), dtype=torch.float32,
            device=feature_map.device, requires_grad=True
        )

        T = torch.ones(B * C, S, dtype=torch.float32, device=feature_map.device) * t_tensor
        x_input = torch.cat([x, T], dim=-1)

        out = self.mlp(x_input)
        u_hat = torch.sigmoid(out.reshape(B, C, H, W, D))

        return u_hat, t_tensor

In [ ]:
#CELL 4 — Laplacian Kernel
def get_laplacian_kernel(device):
    K = torch.zeros(1, 1, 3, 3, 3, device=device)

    K[0, 0, 1, 1, 0] =  1
    K[0, 0, 1, 1, 2] =  1
    K[0, 0, 1, 0, 1] =  1
    K[0, 0, 1, 2, 1] =  1
    K[0, 0, 0, 1, 1] =  1
    K[0, 0, 2, 1, 1] =  1

    K[0, 0, 1, 1, 1] = -6

    return K

In [ ]:
#CELL 5 — PDELoss
class PDELoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, u_hat, du_dt, d, rho):
        B, C, H, W, D = u_hat.shape
        K = get_laplacian_kernel(u_hat.device)

        du = d * u_hat
        du_in = du.reshape(B * C, 1, H, W, D)
        diffusion = F.conv3d(du_in, K, padding=1).reshape(B, C, H, W, D)

        proliferation = rho * u_hat * (1 - u_hat)

        residual = du_dt - diffusion - proliferation
        return (residual ** 2).mean()

In [ ]:
#CELL 6 — Boundary Condition Loss
class BoundaryConditionLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, u_hat, d):
        u = u_hat.float()
        B, C, H, W, D = u.shape

        ux_0 = (u[:, :, 1, :, :]  - u[:, :, 0, :, :])  ** 2
        ux_H = (u[:, :, -1, :, :] - u[:, :, -2, :, :]) ** 2
        uy_0 = (u[:, :, :, 1, :]  - u[:, :, :, 0, :])  ** 2
        uy_W = (u[:, :, :, -1, :] - u[:, :, :, -2, :]) ** 2
        uz_0 = (u[:, :, :, :, 1]  - u[:, :, :, :, 0])  ** 2
        uz_D = (u[:, :, :, :, -1] - u[:, :, :, -2]) ** 2

        bc_loss = (
            ux_0.mean() + ux_H.mean() +
            uy_0.mean() + uy_W.mean() +
            uz_0.mean() + uz_D.mean()
        )

        if isinstance(d, torch.Tensor):
            d_scalar = d.mean()
        else:
            d_scalar = float(d)

        return d_scalar * bc_loss

In [ ]:
#CELL 7 — BiophysicsRegulariser
class BiophysicsRegulariser(nn.Module):
    def __init__(self, in_channels=192, spatial_size=16,
                 hidden_dim=256, lambda1=1e-4, lambda2=1e-4):
        super().__init__()
        self.lambda1 = lambda1
        self.lambda2 = lambda2
        self.d_min,   self.d_max   = 0.02, 1.5
        self.rho_min, self.rho_max = 0.002, 0.2

        self.estimator = TumourCellDensityEstimator(
            in_channels=in_channels,
            spatial_size=spatial_size,
            hidden_dim=hidden_dim,
        )
        self.pde_loss = PDELoss()
        self.bc_loss  = BoundaryConditionLoss()

    def _sample_params_per_voxel(self, shape, device, dtype):
        d   = torch.empty(shape, device=device, dtype=dtype).uniform_(self.d_min, self.d_max)
        rho = torch.empty(shape, device=device, dtype=dtype).uniform_(self.rho_min, self.rho_max)
        return d, rho

    def forward(self, feature_map, seg_loss, t=0.1):
        u_hat, t_tensor = self.estimator(feature_map, t=t)

        du_dt = torch.autograd.grad(
            outputs=u_hat,
            inputs=t_tensor,
            grad_outputs=torch.ones_like(u_hat),
            create_graph=True,
            retain_graph=True,
            allow_unused=True
        )[0]

        if du_dt is None:
            du_dt = torch.zeros_like(u_hat)

        du_dt = torch.tanh(du_dt)
        du_dt = du_dt.expand_as(u_hat)

        d, rho = self._sample_params_per_voxel(u_hat.shape, feature_map.device, feature_map.dtype)

        lpde = self.lambda1 * self.pde_loss(u_hat, du_dt, d=d, rho=rho)
        lbc  = self.lambda2 * self.bc_loss(u_hat, d=d)
        total_loss = seg_loss + lpde + lbc

        info = {
            "loss_seg"  : seg_loss.item(),
            "loss_pde"  : lpde.item(),
            "loss_bc"   : lbc.item(),
            "loss_total": total_loss.item(),
        }
        return total_loss, info